# HTDemucs — Music Source Separation

HTDemucs separates a mixed audio track into 4 stems:
- **Drums**
- **Bass**
- **Other** (instruments)
- **Vocals**

This enables remix, analysis, and creative reuse of existing recordings.

HTDemucs uses a **hybrid transformer** architecture that processes audio in both
the waveform and spectrogram domains simultaneously.

In [ ]:
!pip install demucs torch torchaudio IPython matplotlib numpy scipy

In [ ]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import os
import subprocess

## Create a Sample Mix

We synthesize a simple multi-instrument mix for demonstration.
In practice, you would use a real audio file.

In [ ]:
sr = 44100
duration = 10.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Synthesize simple components
# Bass: low sine with slight wobble
bass_freq = 55.0  # A1
bass = 0.4 * np.sin(2 * np.pi * bass_freq * t) * (1 + 0.3 * np.sin(2 * np.pi * 0.5 * t))

# Drums: periodic impulses with noise burst
kick_times = np.arange(0, duration, 0.5)  # 120 BPM
drums = np.zeros_like(t)
for kt in kick_times:
    idx = int(kt * sr)
    length = min(int(0.1 * sr), len(drums) - idx)
    env = np.exp(-30 * np.linspace(0, 0.1, length))
    drums[idx:idx + length] += 0.5 * np.sin(2 * np.pi * 60 * np.linspace(0, 0.1, length)) * env

# Melody: simple sine melody
melody_notes = [440, 494, 523, 587, 523, 494, 440, 392]  # A4 B4 C5 D5 ...
note_dur = duration / len(melody_notes)
melody = np.zeros_like(t)
for i, freq in enumerate(melody_notes):
    start = int(i * note_dur * sr)
    end = int((i + 1) * note_dur * sr)
    seg_t = np.linspace(0, note_dur, end - start, endpoint=False)
    env = np.minimum(seg_t / 0.02, 1.0) * np.minimum((note_dur - seg_t) / 0.05, 1.0)
    melody[start:end] = 0.3 * np.sin(2 * np.pi * freq * seg_t) * env

# Mix
mix = (bass + drums + melody).astype(np.float32)
mix = mix / np.max(np.abs(mix))  # Normalize

# Save as WAV
import scipy.io.wavfile
scipy.io.wavfile.write("sample.wav", sr, (mix * 32767).astype(np.int16))

print("Sample mix created (10 seconds)")
print("Components: bass (55 Hz), kick drum (120 BPM), melody (A4-D5)")
ipd.display(ipd.Audio(mix, rate=sr))

## Separate Sources

Run HTDemucs to separate the mix into 4 stems.

In [ ]:
# Run HTDemucs for full 4-stem separation
# Output goes to separated/htdemucs/sample/
result = subprocess.run(
    ["python", "-m", "demucs", "--out", "separated", "sample.wav"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("Separation complete!")

## Listen to Separated Stems

In [ ]:
# Find the output directory (demucs names the model subdirectory)
output_dir = None
for model_name in ["htdemucs", "htdemucs_ft", "mdx_extra"]:
    candidate = os.path.join("separated", model_name, "sample")
    if os.path.exists(candidate):
        output_dir = candidate
        break

if output_dir is None:
    # Try to find any output
    for root, dirs, files in os.walk("separated"):
        if any(f.endswith(".wav") for f in files):
            output_dir = root
            break

print(f"Output directory: {output_dir}")

stem_names = ["drums", "bass", "other", "vocals"]
stems = {}

for name in stem_names:
    filepath = os.path.join(output_dir, f"{name}.wav")
    if os.path.exists(filepath):
        waveform, stem_sr = torchaudio.load(filepath)
        stems[name] = waveform.numpy()
        print(f"\n{name.upper()} stem ({waveform.shape[-1] / stem_sr:.1f}s):")
        ipd.display(ipd.Audio(waveform.numpy(), rate=stem_sr))
    else:
        print(f"Warning: {filepath} not found")

In [ ]:
# Visualize spectrograms of each stem
fig, axes = plt.subplots(len(stems) + 1, 1, figsize=(14, 3 * (len(stems) + 1)))

# Original mix
axes[0].specgram(mix, Fs=sr, NFFT=2048, noverlap=1024, cmap="magma")
axes[0].set_title("Original Mix")
axes[0].set_ylabel("Freq (Hz)")
axes[0].set_ylim(0, 8000)

# Each stem
for i, (name, waveform) in enumerate(stems.items()):
    audio_mono = waveform.mean(axis=0) if waveform.ndim > 1 else waveform.squeeze()
    axes[i + 1].specgram(audio_mono, Fs=stem_sr, NFFT=2048, noverlap=1024, cmap="magma")
    axes[i + 1].set_title(f"{name.upper()} Stem")
    axes[i + 1].set_ylabel("Freq (Hz)")
    axes[i + 1].set_ylim(0, 8000)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## Creative Remixing

Once we have separated stems, we can recombine them with different levels,
creating new mixes from the original material.

In [ ]:
def remix_stems(stems, gains, stem_sr):
    """Mix stems together with specified gain levels."""
    # Find minimum length across all stems
    min_len = min(s.shape[-1] for s in stems.values())
    remix = np.zeros(min_len)
    for name, gain in gains.items():
        if name in stems:
            audio = stems[name]
            # Convert to mono if stereo
            mono = audio.mean(axis=0) if audio.ndim > 1 else audio.squeeze()
            remix += gain * mono[:min_len]
    remix = remix / np.max(np.abs(remix) + 1e-8)  # Normalize
    return remix.astype(np.float32)

# Remix 1: Drums + Bass only (karaoke-like)
print("Remix 1: Drums + Bass only")
remix1 = remix_stems(stems, {"drums": 1.0, "bass": 1.0, "other": 0.0, "vocals": 0.0}, stem_sr)
ipd.display(ipd.Audio(remix1, rate=stem_sr))

# Remix 2: Everything except drums
print("\nRemix 2: No drums")
remix2 = remix_stems(stems, {"drums": 0.0, "bass": 1.0, "other": 1.0, "vocals": 1.0}, stem_sr)
ipd.display(ipd.Audio(remix2, rate=stem_sr))

# Remix 3: Boosted bass, quiet everything else
print("\nRemix 3: Bass-heavy mix")
remix3 = remix_stems(stems, {"drums": 0.5, "bass": 2.0, "other": 0.3, "vocals": 0.3}, stem_sr)
ipd.display(ipd.Audio(remix3, rate=stem_sr))

## Evaluation

How well does the separation work? We compare the original mix to the
sum of all stems to check for artifacts.

In [ ]:
# Reconstruct by summing all stems
reconstructed = remix_stems(stems, {"drums": 1.0, "bass": 1.0, "other": 1.0, "vocals": 1.0}, stem_sr)

# Compare original and reconstructed
# Resample original to match if needed
if sr != stem_sr:
    mix_resampled = torchaudio.functional.resample(
        torch.tensor(mix).unsqueeze(0), sr, stem_sr
    ).numpy().squeeze()
else:
    mix_resampled = mix

min_len = min(len(mix_resampled), len(reconstructed))
original_segment = mix_resampled[:min_len]
reconstructed_segment = reconstructed[:min_len]

# Normalize both for comparison
original_segment = original_segment / (np.max(np.abs(original_segment)) + 1e-8)
reconstructed_segment = reconstructed_segment / (np.max(np.abs(reconstructed_segment)) + 1e-8)

# Compute difference
difference = original_segment - reconstructed_segment

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

time = np.arange(min_len) / stem_sr
axes[0].plot(time, original_segment, linewidth=0.5)
axes[0].set_title("Original Mix")
axes[0].set_ylabel("Amplitude")

axes[1].plot(time, reconstructed_segment, linewidth=0.5, color="orange")
axes[1].set_title("Sum of Separated Stems")
axes[1].set_ylabel("Amplitude")

axes[2].plot(time, difference, linewidth=0.5, color="red")
axes[2].set_title(f"Difference (RMS: {np.sqrt(np.mean(difference**2)):.4f})")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

print(f"Reconstruction RMS error: {np.sqrt(np.mean(difference**2)):.6f}")
print(f"Reconstruction SNR: {10 * np.log10(np.mean(original_segment**2) / (np.mean(difference**2) + 1e-10)):.1f} dB")